# Prompt 32 — Hands-On Capstone Project
## Ride-Sharing Pipeline: All 7 Exam Topic Areas in One Cohesive Workflow

**Exam Coverage**: This notebook exercises every topic area on the Databricks Certified Associate Developer for Apache Spark exam:

| Step | Topic Area | Exam Weight |
|------|-----------|-------------|
| 1–2 | Spark Architecture + DataFrame API (Schema / Inspect) | 20% + 30% |
| 3 | DataFrame API — Type Casting | 30% |
| 4 | DataFrame API — Null Handling | 30% |
| 5 | DataFrame API — Filter / Sort | 30% |
| 6 | Spark SQL + DataFrame API — Aggregations | 20% + 30% |
| 7 | Spark SQL — Window Functions | 20% |
| 8 | DataFrame API — UDFs | 30% |
| 9 | Spark SQL — Temp Views + SQL Queries | 20% |
| 10 | DataFrame API — Broadcast Join | 30% |
| 11 | Troubleshooting & Tuning — Parquet / Partition Pruning | 10% |
| 12 | Structured Streaming | 10% |
| Scenarios | All topics extended | All |

---

**Dataset**: Ride-sharing trip data created in-memory with columns:
`trip_id`, `driver_id`, `rider_id`, `pickup_time` (string), `dropoff_time` (string),
`fare_amount`, `distance_miles`, `status`, `city`, `tip_amount` (some nulls)

## Setup — SparkSession

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("RideSharingCapstone")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

---
## Step 1 — Define an Explicit StructType Schema and Create the DataFrame

> **Exam concept**: Using an explicit `StructType` schema guarantees column names and types on creation — avoids schema inference overhead and prevents silent type mismatches.

In [ ]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType
)

trip_schema = StructType([
    StructField("trip_id",        StringType(), nullable=False),
    StructField("driver_id",      StringType(), nullable=False),
    StructField("rider_id",       StringType(), nullable=False),
    StructField("pickup_time",    StringType(), nullable=True),   # will be cast
    StructField("dropoff_time",   StringType(), nullable=True),   # will be cast
    StructField("fare_amount",    DoubleType(), nullable=True),   # some rows null
    StructField("distance_miles", DoubleType(), nullable=False),
    StructField("status",         StringType(), nullable=False),
    StructField("city",           StringType(), nullable=False),
    StructField("tip_amount",     DoubleType(), nullable=True),   # intentional nulls
])

trip_data = [
    ("T001", "D001", "R001", "2024-03-01 08:00:00", "2024-03-01 08:25:00", 22.50, 5.2,  "completed",  "NYC",   3.00),
    ("T002", "D002", "R002", "2024-03-01 09:10:00", "2024-03-01 09:45:00", 35.00, 8.1,  "completed",  "LA",    5.00),
    ("T003", "D001", "R003", "2024-03-01 10:00:00", "2024-03-01 10:15:00",  8.00, 1.9,  "completed",  "NYC",   None),
    ("T004", "D003", "R004", "2024-03-01 11:00:00", "2024-03-01 11:40:00", 42.00, 10.5, "completed",  "Chicago", 7.00),
    ("T005", "D002", "R005", "2024-03-01 12:00:00", "2024-03-01 12:10:00",  None, 2.0,  "cancelled",  "LA",    None),
    ("T006", "D004", "R006", "2024-03-01 13:00:00", "2024-03-01 13:30:00", 28.75, 6.3,  "completed",  "NYC",   None),
    ("T007", "D003", "R007", "2024-03-01 14:15:00", "2024-03-01 15:00:00", 55.00, 13.0, "completed",  "Chicago", 10.00),
    ("T008", "D005", "R008", "2024-03-01 15:00:00", "2024-03-01 15:20:00", 14.50, 3.1,  "completed",  "SF",    2.00),
    ("T009", "D001", "R009", "2024-03-01 16:00:00", "2024-03-01 16:45:00", 31.00, 7.2,  "completed",  "NYC",   4.00),
    ("T010", "D004", "R010", "2024-03-01 17:00:00", "2024-03-01 17:15:00",  9.50, 1.8,  "cancelled",  "NYC",   None),
    ("T011", "D005", "R011", "2024-03-01 18:00:00", "2024-03-01 18:50:00", 47.00, 11.2, "completed",  "SF",    8.00),
    ("T012", "D002", "R012", "2024-03-01 19:00:00", "2024-03-01 19:20:00", 18.00, 4.0,  "completed",  "LA",    3.00),
    ("T013", "D006", "R013", "2024-03-01 20:00:00", "2024-03-01 20:30:00", 25.00, 5.8,  "completed",  "Chicago", None),
    ("T014", "D006", "R014", "2024-03-01 21:00:00", "2024-03-01 21:45:00", 38.50, 9.0,  "completed",  "Chicago", 6.00),
    ("T015", "D007", "R015", "2024-03-01 22:00:00", "2024-03-01 22:25:00",  7.00, 1.5,  "completed",  "NYC",    1.00),
    ("T016", "D007", "R016", "2024-03-02 07:00:00", "2024-03-02 07:40:00", 29.00, 6.8,  "completed",  "NYC",   None),
    ("T017", "D008", "R017", "2024-03-02 08:00:00", "2024-03-02 08:35:00", 33.00, 7.5,  "completed",  "LA",    5.50),
    ("T018", "D008", "R018", "2024-03-02 09:00:00", "2024-03-02 09:10:00",  6.50, 1.2,  "completed",  "LA",    None),
    ("T019", "D009", "R019", "2024-03-02 10:00:00", "2024-03-02 10:50:00", 51.00, 12.1, "completed",  "SF",    9.00),
    ("T020", "D009", "R020", "2024-03-02 11:00:00", "2024-03-02 11:20:00", 21.50, 4.9,  "completed",  "SF",    3.50),
]

trips_df = spark.createDataFrame(trip_data, schema=trip_schema)

print(f"Row count: {trips_df.count()}")
print(f"Column count: {len(trips_df.columns)}")

**Expected output:**
```
Row count: 20
Column count: 10
```

> **Exam concept**: `StructType` + `StructField` defines explicit schemas. `nullable=True` marks optional columns. Schema-first creation avoids costly inference on large datasets.

---
## Step 2 — Inspect with `printSchema()` and `show()`; Identify Issues

> **Exam concept**: Use `printSchema()` to verify data types; use `show()` to spot null values, string timestamps, and unexpected values before transforming.

In [ ]:
# --- Schema inspection ---
trips_df.printSchema()

**Expected output:**
```
root
 |-- trip_id: string (nullable = false)
 |-- driver_id: string (nullable = false)
 |-- rider_id: string (nullable = false)
 |-- pickup_time: string (nullable = true)
 |-- dropoff_time: string (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- distance_miles: double (nullable = false)
 |-- status: string (nullable = false)
 |-- city: string (nullable = false)
 |-- tip_amount: double (nullable = true)
```

In [ ]:
# --- Row preview ---
trips_df.show(5, truncate=False)

**Expected output (first 5 rows):**
```
+-------+---------+-------+-------------------+-------------------+-----------+--------------+---------+-------+----------+
|trip_id|driver_id|rider_id|pickup_time        |dropoff_time       |fare_amount|distance_miles|status   |city   |tip_amount|
+-------+---------+-------+-------------------+-------------------+-----------+--------------+---------+-------+----------+
|T001   |D001     |R001   |2024-03-01 08:00:00|2024-03-01 08:25:00|22.5       |5.2           |completed|NYC    |3.0       |
|T002   |D002     |R002   |2024-03-01 09:10:00|2024-03-01 09:45:00|35.0       |8.1           |completed|LA     |5.0       |
|T003   |D001     |R003   |2024-03-01 10:00:00|2024-03-01 10:15:00|8.0        |1.9           |completed|NYC    |null      |
|T004   |D003     |R004   |2024-03-01 11:00:00|2024-03-01 11:40:00|42.0       |10.5          |completed|Chicago|7.0       |
|T005   |D002     |R005   |2024-03-01 12:00:00|2024-03-01 12:10:00|null       |2.0           |cancelled|LA     |null      |
+-------+---------+-------+-------------------+-------------------+-----------+--------------+---------+-------+----------+
```

In [ ]:
from pyspark.sql import functions as F

# --- Count nulls in each column ---
null_counts = trips_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in trips_df.columns
])
null_counts.show()

print("Issues identified:")
print("  1. pickup_time and dropoff_time are StringType — need casting to TimestampType")
print("  2. tip_amount has nulls — need to fill with 0.0")
print("  3. fare_amount has 1 null (T005) — row should be dropped")

**Expected null counts:**
```
+-------+---------+--------+-----------+------------+-----------+--------------+------+----+----------+
|trip_id|driver_id|rider_id|pickup_time|dropoff_time|fare_amount|distance_miles|status|city|tip_amount|
+-------+---------+--------+-----------+------------+-----------+--------------+------+----+----------+
|      0|        0|       0|          0|           0|          1|             0|     0|   0|         8|
+-------+---------+--------+-----------+------------+-----------+--------------+------+----+----------+
```

> **Exam concept**: Inspecting nulls programmatically with `isNull()` + `cast("int")` + `sum()` is the standard pattern for data quality checks in PySpark.

---
## Step 3 — Cast `pickup_time` and `dropoff_time` to `TimestampType`

> **Exam concept**: Use `col().cast(TimestampType())` or `to_timestamp()` to convert string columns. Using `withColumn()` preserves all other columns and replaces only the target column.

In [ ]:
from pyspark.sql.types import TimestampType

trips_df = (
    trips_df
    .withColumn("pickup_time",  F.to_timestamp("pickup_time",  "yyyy-MM-dd HH:mm:ss"))
    .withColumn("dropoff_time", F.to_timestamp("dropoff_time", "yyyy-MM-dd HH:mm:ss"))
)

# Verify the types changed
trips_df.printSchema()

**Expected output (relevant lines):**
```
 |-- pickup_time: timestamp (nullable = true)
 |-- dropoff_time: timestamp (nullable = true)
```

In [ ]:
# Compute trip duration in minutes as a derived column
trips_df = trips_df.withColumn(
    "duration_minutes",
    (F.unix_timestamp("dropoff_time") - F.unix_timestamp("pickup_time")) / 60
)

trips_df.select("trip_id", "pickup_time", "dropoff_time", "duration_minutes").show(5, truncate=False)

**Expected output:**
```
+-------+-------------------+-------------------+----------------+
|trip_id|pickup_time        |dropoff_time       |duration_minutes|
+-------+-------------------+-------------------+----------------+
|T001   |2024-03-01 08:00:00|2024-03-01 08:25:00|25.0            |
|T002   |2024-03-01 09:10:00|2024-03-01 09:45:00|35.0            |
|T003   |2024-03-01 10:00:00|2024-03-01 10:15:00|15.0            |
|T004   |2024-03-01 11:00:00|2024-03-01 11:40:00|40.0            |
|T005   |2024-03-01 12:00:00|2024-03-01 12:10:00|10.0            |
+-------+-------------------+-------------------+----------------+
```

> **Exam concept**: `to_timestamp(col, format)` is preferred over `.cast(TimestampType())` when the string format is known — prevents null coercion on mismatched formats. `unix_timestamp()` enables arithmetic on timestamps.

---
## Step 4 — Fill Null `tip_amount`; Drop Rows Where `fare_amount` is Null

> **Exam concept**: `fillna()` replaces nulls with a default; `dropna()` removes rows with nulls in specified columns. Both return new DataFrames — Spark is immutable.

In [ ]:
print(f"Row count before null handling: {trips_df.count()}")

# Fill null tip_amount with 0.0
trips_df = trips_df.fillna({"tip_amount": 0.0})

# Drop rows where fare_amount is null
trips_df = trips_df.dropna(subset=["fare_amount"])

print(f"Row count after null handling: {trips_df.count()}")

# Confirm no remaining nulls in tip_amount or fare_amount
trips_df.select(
    F.sum(F.col("tip_amount").isNull().cast("int")).alias("tip_nulls"),
    F.sum(F.col("fare_amount").isNull().cast("int")).alias("fare_nulls")
).show()

**Expected output:**
```
Row count before null handling: 20
Row count after null handling: 19
+---------+----------+
|tip_nulls|fare_nulls|
+---------+----------+
|        0|         0|
+---------+----------+
```

> **Exam concept**: `fillna(dict)` targets specific columns — safer than filling all columns. `dropna(subset=[...])` limits removal to rows missing values in the specified set. One row (T005) is removed for null `fare_amount`.

---
## Step 5 — Filter for Completed Trips Only; Sort by `fare_amount` Descending

> **Exam concept**: `filter()` / `where()` apply predicates (narrow transformations — no shuffle). `orderBy()` / `sort()` are wide transformations that trigger a sort stage.

In [ ]:
completed_df = (
    trips_df
    .filter(F.col("status") == "completed")
    .orderBy(F.col("fare_amount").desc())
)

print(f"Completed trips: {completed_df.count()}")
completed_df.select("trip_id", "driver_id", "city", "fare_amount", "status").show(10, truncate=False)

**Expected output:**
```
Completed trips: 17
+-------+---------+-------+-----------+---------+
|trip_id|driver_id|city   |fare_amount|status   |
+-------+---------+-------+-----------+---------+
|T007   |D003     |Chicago|55.0       |completed|
|T019   |D009     |SF     |51.0       |completed|
|T011   |D005     |SF     |47.0       |completed|
|T004   |D003     |Chicago|42.0       |completed|
|T014   |D006     |Chicago|38.5       |completed|
...
```

> **Exam concept**: Cancelled trips (T005, T010) are excluded. `filter()` is a **narrow transformation** — it runs in the same stage as the preceding step. `orderBy()` is a **wide transformation** — it shuffles data to sort globally across partitions.

---
## Step 6 — Aggregate per Driver: Total Trips, Revenue, Avg Fare, Avg Distance

> **Exam concept**: `groupBy().agg()` returns a new DataFrame. Multiple aggregations in a single `agg()` call avoid repeated shuffles. The result type is always a DataFrame.

In [ ]:
driver_stats_df = (
    completed_df
    .groupBy("driver_id", "city")
    .agg(
        F.count("trip_id").alias("total_trips"),
        F.round(F.sum("fare_amount"), 2).alias("total_revenue"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("distance_miles"), 2).alias("avg_distance")
    )
    .orderBy(F.col("total_revenue").desc())
)

driver_stats_df.show(truncate=False)

**Expected output:**
```
+---------+-------+-----------+-------------+--------+------------+
|driver_id|city   |total_trips|total_revenue|avg_fare|avg_distance|
+---------+-------+-----------+-------------+--------+------------+
|D003     |Chicago|2          |97.0         |48.5    |11.75       |
|D009     |SF     |2          |72.5         |36.25   |8.5         |
|D005     |SF     |2          |61.5         |30.75   |7.15        |
|D006     |Chicago|2          |63.5         |31.75   |7.4         |
|D001     |NYC    |3          |61.5         |20.5    |4.77        |
...
```

> **Exam concept**: `groupBy().agg()` triggers a shuffle. Bundling all aggregation expressions in a single `agg()` call produces one shuffle stage instead of many. `F.count("trip_id")` counts non-null values; `F.count("*")` counts all rows.

---
## Step 7 — Rank Drivers by Total Revenue Within Each City Using a Window Function

> **Exam concept**: Window functions use `Window.partitionBy().orderBy()`. `rank()` leaves gaps after ties; `dense_rank()` does not. `row_number()` is unique per row regardless of ties.

In [ ]:
from pyspark.sql.window import Window

city_revenue_window = (
    Window
    .partitionBy("city")
    .orderBy(F.col("total_revenue").desc())
)

ranked_df = driver_stats_df.withColumn(
    "city_rank", F.dense_rank().over(city_revenue_window)
)

ranked_df.select("city", "driver_id", "total_revenue", "city_rank") \
         .orderBy("city", "city_rank") \
         .show(truncate=False)

**Expected output:**
```
+-------+---------+-------------+---------+
|city   |driver_id|total_revenue|city_rank|
+-------+---------+-------------+---------+
|Chicago|D003     |97.0         |1        |
|Chicago|D006     |63.5         |2        |
|LA     |D002     |53.0         |1        |
|LA     |D008     |39.5         |2        |
|NYC    |D001     |61.5         |1        |
|NYC    |D004     |28.75        |2        |
|NYC    |D007     |36.0         |3        |
|SF     |D009     |72.5         |1        |
|SF     |D005     |61.5         |2        |
+-------+---------+-------------+---------+
```

> **Exam concept**: `dense_rank()` produces consecutive ranks within each city partition. Without `partitionBy()`, the window would rank globally. The window spec does NOT trigger a separate DataFrame action — it is evaluated lazily as part of the next action.

---
## Step 8 — UDF: Categorise `fare_amount` into Low / Medium / High

> **Exam concept**: UDFs are black-box functions — the Catalyst optimizer cannot inspect or optimize them. They bypass predicate pushdown and vectorized execution. Prefer built-in functions when possible; use UDFs only for custom logic unavailable natively.

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def categorise_fare(fare):
    if fare is None:
        return "unknown"
    elif fare < 10.0:
        return "low"
    elif fare <= 30.0:
        return "medium"
    else:
        return "high"

categorise_fare_udf = udf(categorise_fare, StringType())

ranked_df = ranked_df.withColumn(
    "fare_category", categorise_fare_udf(F.col("avg_fare"))
)

ranked_df.select("driver_id", "city", "avg_fare", "fare_category").show(truncate=False)

**Expected output:**
```
+---------+-------+--------+-------------+
|driver_id|city   |avg_fare|fare_category|
+---------+-------+--------+-------------+
|D003     |Chicago|48.5    |high         |
|D006     |Chicago|31.75   |high         |
|D002     |LA     |26.5    |medium       |
|D008     |LA     |19.75   |medium       |
|D001     |NYC    |20.5    |medium       |
|D004     |NYC    |28.75   |medium       |
|D007     |NYC    |18.0    |medium       |
|D009     |SF     |36.25   |high         |
|D005     |SF     |30.75   |high         |
+---------+-------+--------+-------------+
```

> **Exam concept**: Always check for `None` inside the UDF body — Python UDFs receive `None` for null SQL values. Not handling `None` raises a `NullPointerException` or returns unexpected results. Register with `@udf(returnType=StringType())` decorator or `udf(fn, returnType)` factory.

---
## Step 9 — Register as Temp View; Query with `spark.sql()` for Top 5 Drivers by Revenue

> **Exam concept**: `createOrReplaceTempView()` makes a DataFrame queryable via SQL. The view lives in the SparkSession scope. `spark.sql()` returns a DataFrame — all subsequent DataFrame transformations apply.

In [ ]:
# Register the ranked DataFrame as a temp view
ranked_df.createOrReplaceTempView("driver_rankings")

# Query via spark.sql() — returns a DataFrame
top5_sql = spark.sql("""
    SELECT
        driver_id,
        city,
        total_trips,
        total_revenue,
        avg_fare,
        fare_category,
        city_rank
    FROM driver_rankings
    ORDER BY total_revenue DESC
    LIMIT 5
""")

top5_sql.show(truncate=False)

**Expected output:**
```
+---------+-------+-----------+-------------+--------+-------------+---------+
|driver_id|city   |total_trips|total_revenue|avg_fare|fare_category|city_rank|
+---------+-------+-----------+-------------+--------+-------------+---------+
|D003     |Chicago|2          |97.0         |48.5    |high         |1        |
|D009     |SF     |2          |72.5         |36.25   |high         |1        |
|D006     |Chicago|2          |63.5         |31.75   |high         |2        |
|D001     |NYC    |3          |61.5         |20.5    |medium       |1        |
|D005     |SF     |2          |61.5         |30.75   |high         |2        |
+---------+-------+-----------+-------------+--------+-------------+---------+
```

> **Exam concept**: `createOrReplaceTempView` is session-scoped — it disappears when the SparkSession ends. `createGlobalTempView` is application-scoped, accessible across sessions via `global_temp.view_name`. `spark.sql()` supports full SQL syntax including subqueries, CTEs, and window functions.

---
## Step 10 — Broadcast Join with `driver_details` DataFrame

> **Exam concept**: Wrap the smaller DataFrame with `F.broadcast()` to hint Spark to use a `BroadcastHashJoin` instead of a `SortMergeJoin`. This eliminates the shuffle on the larger side. Threshold: `spark.sql.autoBroadcastJoinThreshold` (default 10 MB).

In [ ]:
from pyspark.sql.functions import broadcast

# Small lookup table — ideal broadcast candidate
driver_details_data = [
    ("D001", "Alice Johnson",  4.9),
    ("D002", "Bob Smith",      4.7),
    ("D003", "Carol Davis",    4.8),
    ("D004", "David Lee",      4.5),
    ("D005", "Eva Martinez",   4.6),
    ("D006", "Frank Wilson",   4.3),
    ("D007", "Grace Kim",      4.8),
    ("D008", "Henry Brown",    4.4),
    ("D009", "Iris Chen",      4.9),
]

driver_details_schema = StructType([
    StructField("driver_id", StringType(), nullable=False),
    StructField("name",      StringType(), nullable=False),
    StructField("rating",    DoubleType(), nullable=False),
])

driver_details_df = spark.createDataFrame(driver_details_data, schema=driver_details_schema)

# Broadcast join — driver_details is small, so broadcast it
enriched_df = (
    top5_sql
    .join(broadcast(driver_details_df), on="driver_id", how="left")
    .select("driver_id", "name", "rating", "city", "total_revenue", "fare_category")
)

enriched_df.show(truncate=False)

In [ ]:
# Confirm BroadcastHashJoin appears in the physical plan
enriched_df.explain(mode="formatted")

**Expected output (enriched_df):**
```
+---------+-------------+------+-------+-------------+-------------+
|driver_id|name         |rating|city   |total_revenue|fare_category|
+---------+-------------+------+-------+-------------+-------------+
|D003     |Carol Davis  |4.8   |Chicago|97.0         |high         |
|D009     |Iris Chen    |4.9   |SF     |72.5         |high         |
|D006     |Frank Wilson |4.3   |Chicago|63.5         |high         |
|D001     |Alice Johnson|4.9   |NYC    |61.5         |medium       |
|D005     |Eva Martinez |4.6   |SF     |61.5         |high         |
+---------+-------------+------+-------+-------------+-------------+
```
**Expected explain() (key line):**
```
BroadcastHashJoin [driver_id], [driver_id], LeftOuter, BuildRight
```

> **Exam concept**: The `broadcast()` hint overrides the `autoBroadcastJoinThreshold` size check. The larger DataFrame (left side) stays in place; the broadcast table is replicated to every executor — zero shuffle on the left side. Set `spark.sql.autoBroadcastJoinThreshold = -1` to disable auto-broadcast.

---
## Step 11 — Write to Parquet Partitioned by City; Read Back; Confirm Partition Pruning

> **Exam concept**: Writing with `partitionBy("city")` creates subdirectories per value. Reading with a filter on the partition key triggers **partition pruning** — Spark skips directories that don't match, reducing I/O without touching every file.

In [ ]:
import tempfile, os

# Use a temp directory for the Parquet output
parquet_path = os.path.join(tempfile.gettempdir(), "capstone_driver_rankings")

# Write partitioned by city
ranked_df.write \
    .mode("overwrite") \
    .partitionBy("city") \
    .parquet(parquet_path)

print(f"Written to: {parquet_path}")

# List the partition directories created
for entry in sorted(os.listdir(parquet_path)):
    print(" ", entry)

**Expected output:**
```
Written to: /tmp/capstone_driver_rankings
  city=Chicago
  city=LA
  city=NYC
  city=SF
  _SUCCESS
```

In [ ]:
# Read back and filter — Spark will prune to city=NYC only
pruned_df = (
    spark.read.parquet(parquet_path)
    .filter(F.col("city") == "NYC")
)

pruned_df.show(truncate=False)

In [ ]:
# Confirm partition pruning in the physical plan
# Look for: PartitionFilters: [isnotnull(city#...), (city#... = NYC)]
pruned_df.explain(mode="formatted")

**Expected explain() output (key section):**
```
== Physical Plan ==
...
FileScan parquet [driver_id#..., city#...]
   PartitionFilters: [isnotnull(city#...), (city#... = NYC)]
   PushedFilters: []
   ReadSchema: ...
```

> **Exam concept**: `PartitionFilters` in `explain()` confirms Spark reads only the `city=NYC` directory. Without partitioning, Spark would scan all files. Partition pruning is automatic when the filter column matches the partition key — no explicit code required beyond writing with `partitionBy()`.

---
## Step 12 — Structured Streaming: Running Fare Total per City from a Rate Source

> **Exam concept**: Structured Streaming treats a live data stream as an unbounded table. The same DataFrame API applies. A **rate source** generates rows at a fixed rate — ideal for testing without an external system.

In [ ]:
# Rate source: generates (timestamp, value) rows at 1 row/second
# We simulate fare data by mapping value to a synthetic fare and city

cities = ["NYC", "LA", "Chicago", "SF"]

rate_stream_df = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 4)
    .load()
)

# Derive synthetic trip fields from the auto-incrementing value
streaming_trips_df = rate_stream_df.select(
    F.col("timestamp"),
    (F.col("value") % 4).cast("int").alias("city_idx"),
    ((F.col("value") % 50) + 5).cast("double").alias("fare_amount")
).withColumn(
    "city",
    F.element_at(F.array([F.lit(c) for c in cities]), (F.col("city_idx") + 1).cast("int"))
)

# Aggregate: running total fare per city
fare_totals_df = (
    streaming_trips_df
    .groupBy("city")
    .agg(F.sum("fare_amount").alias("running_fare_total"))
)

print("Streaming DataFrame schema:")
fare_totals_df.printSchema()

**Expected schema output:**
```
Streaming DataFrame schema:
root
 |-- city: string (nullable = true)
 |-- running_fare_total: double (nullable = true)
```

In [ ]:
import time

# Write to memory sink for easy inspection; complete mode required for groupBy without watermark
streaming_query = (
    fare_totals_df
    .writeStream
    .format("memory")
    .queryName("fare_totals")
    .outputMode("complete")    # complete = full result table every micro-batch
    .trigger(processingTime="2 seconds")
    .start()
)

print("Streaming query started. Waiting 8 seconds for data...")
time.sleep(8)

# Query the in-memory result table
spark.sql("SELECT * FROM fare_totals ORDER BY running_fare_total DESC").show()

# Stop the stream cleanly
streaming_query.stop()
print("Streaming query stopped.")

**Expected output (values will vary with timing):**
```
Streaming query started. Waiting 8 seconds for data...
+-------+------------------+
|city   |running_fare_total|
+-------+------------------+
|Chicago|348.0             |
|SF     |346.0             |
|NYC    |344.0             |
|LA     |342.0             |
+-------+------------------+
Streaming query stopped.
```

> **Exam concept**:  
> - **`complete` output mode** re-emits the entire result table every trigger — required for stateful aggregations without a watermark.  
> - **`memory` sink** stores results in an in-memory table queryable via `spark.sql()` — useful only for testing.  
> - `streaming_query.stop()` is essential to release executor resources; without it the query runs indefinitely.  
> - `processingTime` trigger controls micro-batch frequency. `Trigger.Once()` processes all available data and stops.

---
## Full Pipeline Summary

| Step | Operation | Key API | Exam Topic |
|------|-----------|---------|------------|
| 1 | Schema + DataFrame creation | `StructType`, `createDataFrame` | DataFrame API |
| 2 | Inspect schema and nulls | `printSchema`, `show`, `isNull` | DataFrame API |
| 3 | Type casting | `to_timestamp`, `withColumn` | DataFrame API |
| 4 | Null handling | `fillna`, `dropna` | DataFrame API |
| 5 | Filter + sort | `filter`, `orderBy` | DataFrame API |
| 6 | Aggregations | `groupBy`, `agg`, `sum`, `avg` | Spark SQL |
| 7 | Window ranking | `Window`, `dense_rank` | Spark SQL |
| 8 | UDF application | `udf`, `withColumn` | DataFrame API |
| 9 | Temp view + SQL query | `createOrReplaceTempView`, `spark.sql` | Spark SQL |
| 10 | Broadcast join | `broadcast`, `join` | DataFrame API |
| 11 | Parquet write + partition pruning | `partitionBy`, `explain` | Tuning |
| 12 | Structured Streaming | `readStream`, `writeStream`, `complete` | Streaming |

---
## Real-World Extension Scenarios

<details>
<summary>Scenario 1 — Late-Arriving Records Break the Streaming Fare Totals</summary>

**Problem description:**
The production streaming pipeline for running fare totals uses `complete` output mode. Operations staff report that trips completed 15–30 minutes before the current micro-batch are still arriving from a slow mobile client. Without watermarking, Spark keeps all state indefinitely, causing executor heap memory to grow without bound in long-running jobs.

**Root cause:**
- `complete` output mode requires retaining the entire aggregation state — it never discards state.
- Late-arriving records extend the state even further.
- For a long-running production stream, this causes unbounded state growth and eventual OOM errors.

**PySpark resolution steps:**
```python
# Switch to update output mode + add watermark to bound state
streaming_trips_with_wm = (
    streaming_trips_df
    .withWatermark("timestamp", "30 minutes")  # discard events > 30 min late
)

windowed_fare = (
    streaming_trips_with_wm
    .groupBy(
        F.window("timestamp", "5 minutes"),   # tumbling 5-min window
        F.col("city")
    )
    .agg(F.sum("fare_amount").alias("window_fare_total"))
)

query = (
    windowed_fare
    .writeStream
    .format("memory")
    .queryName("windowed_fares")
    .outputMode("update")   # only emit updated windows
    .start()
)
```

**Expected outcome:**
- Events older than 30 minutes relative to the watermark are silently dropped.
- State is bounded — Spark discards expired windows, preventing OOM.
- `update` mode emits only the windows that were updated in the current micro-batch, reducing output volume.

**Exam topic reinforced:** Structured Streaming — watermarking, output modes, state management.

</details>

<details>
<summary>Scenario 2 — Corrupt Parquet Partition Blocks the Daily Report Job</summary>

**Problem description:**
The nightly job that reads `capstone_driver_rankings` partitioned by city fails with:
```
SparkException: Malformed records are detected in record parsing.
org.apache.spark.sql.execution.datasources.SchemaColumnConvertNotSupportedException
```
Investigation shows that `city=NYC` was written by a different job using a slightly different schema — `fare_category` was written as `IntegerType` instead of `StringType`.

**PySpark resolution steps:**
```python
# Option 1: Use schema merging with PERMISSIVE mode to skip corrupt records
safe_df = (
    spark.read
    .option("mergeSchema", "true")
    .option("mode", "PERMISSIVE")
    .parquet(parquet_path)
)

# Option 2: Read only non-corrupt partitions explicitly
safe_df = spark.read.parquet(
    f"{parquet_path}/city=Chicago",
    f"{parquet_path}/city=LA",
    f"{parquet_path}/city=SF"
)

# Option 3: Rewrite the corrupt partition from source data
nyc_repaired = ranked_df.filter(F.col("city") == "NYC")
nyc_repaired.write \
    .mode("overwrite") \
    .parquet(f"{parquet_path}/city=NYC")
```

**Expected outcome:**
- PERMISSIVE mode allows the read to succeed; corrupt rows appear with `null` in mismatched columns.
- Rewriting the partition from canonical source data is the cleanest resolution.
- `mergeSchema` is useful when columns are added over time but can't resolve type conflicts.

**Exam topic reinforced:** Troubleshooting & Tuning — schema mismatch errors, Parquet partition repair, `PERMISSIVE` read mode.

</details>

<details>
<summary>Scenario 3 — Data Skew in Driver Rankings Causes One Task to Run 20× Longer</summary>

**Problem description:**
In a production dataset with millions of trips, `groupBy("driver_id", "city")` in Step 6 causes one Spark task to run 20× longer than the median. Spark UI shows the task processing the `NYC` partition handles 65% of all rows because D001 has far more trips than any other driver.

**Diagnostic steps:**
```python
# Check trip distribution — spot skewed driver
trips_df.groupBy("driver_id").count().orderBy(F.col("count").desc()).show()

# In the Spark UI: Stages > Tasks > sort by Duration DESC
# A skewed task shows: Duration=120s, median=6s
```

**PySpark resolution — salting:**
```python
import random

SALT_FACTOR = 10

# Add random salt key to the large DataFrame
salted_trips = trips_df.withColumn(
    "salt", (F.rand() * SALT_FACTOR).cast("int")
)

# Pre-aggregate with salt to reduce data per key
pre_agg = (
    salted_trips
    .groupBy("driver_id", "city", "salt")
    .agg(
        F.count("trip_id").alias("partial_trips"),
        F.sum("fare_amount").alias("partial_revenue")
    )
)

# Final aggregation without salt
final_agg = (
    pre_agg
    .groupBy("driver_id", "city")
    .agg(
        F.sum("partial_trips").alias("total_trips"),
        F.sum("partial_revenue").alias("total_revenue")
    )
)

# AQE alternative (no code change needed — just enable AQE)
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
```

**Expected outcome:**
- Salting distributes the skewed key across 10 sub-keys, balancing task sizes.
- AQE (when enabled) auto-detects skew at runtime and splits skewed partitions — no code change required.
- Task duration variance in Spark UI drops from 20:1 to under 2:1.

**Exam topic reinforced:** Troubleshooting & Tuning — data skew detection, salting technique, AQE skew join optimization.

</details>

<details>
<summary>Scenario 4 — UDF Performance Bottleneck: Replace with Built-In Functions</summary>

**Problem description:**
The fare categorisation UDF in Step 8 is fast on small data but becomes the slowest stage in production because:
1. UDFs bypass Catalyst optimizer — no code generation or predicate pushdown.
2. Data must be deserialized from JVM objects to Python (via Py4J), processed, then re-serialized — significant CPU overhead per row.
3. The Spark UI shows the UDF stage taking 40% of total job time on 100M rows.

**PySpark resolution — replace UDF with `when/otherwise`:**
```python
from pyspark.sql.functions import when

# Built-in equivalent — Catalyst-optimized, no Python serialization
categorised_df = ranked_df.withColumn(
    "fare_category",
    when(F.col("avg_fare") < 10.0,                 F.lit("low"))
    .when(F.col("avg_fare").between(10.0, 30.0),   F.lit("medium"))
    .otherwise(                                     F.lit("high"))
)

# For complex logic that still needs Python: use pandas_udf (vectorized)
from pyspark.sql.functions import pandas_udf
import pandas as pd

@pandas_udf(StringType())
def categorise_fare_vectorized(fare_series: pd.Series) -> pd.Series:
    return fare_series.apply(
        lambda f: "low" if f < 10 else ("medium" if f <= 30 else "high")
        if pd.notna(f) else "unknown"
    )

vectorized_df = ranked_df.withColumn(
    "fare_category", categorise_fare_vectorized(F.col("avg_fare"))
)
```

**Expected outcome:**
- `when/otherwise` runs entirely in the JVM using generated bytecode — typically 5–10× faster than a Python UDF.
- `pandas_udf` uses Apache Arrow for batch serialization — typically 2–5× faster than a row-level Python UDF.
- UDF stage disappears from Spark UI bottleneck analysis.

**Exam topic reinforced:** DataFrame API — UDFs vs built-in functions; Troubleshooting & Tuning — performance impact of UDFs.

</details>

<details>
<summary>Scenario 5 — Spark Connect Client Crash Doesn't Kill the Cluster Job</summary>

**Problem description:**
The data engineering team migrates the capstone pipeline to use Spark Connect so analysts can run it from their laptops via a thin Python client. During a long-running Step 11 Parquet write, an analyst's laptop loses network connectivity. In the classic Spark deployment, this would kill the driver process and abort the job. The team wants to understand whether the job survives the client disconnect with Spark Connect.

**Architecture explanation:**
```python
# Classic Spark: driver runs IN the client process
# Client crash = driver crash = job aborted
spark_classic = SparkSession.builder \
    .appName("Classic") \
    .getOrCreate()

# Spark Connect: driver runs ON the cluster; client is a thin stub
# Client disconnect = client loses connection; driver continues on cluster
spark_connect = SparkSession.builder \
    .remote("sc://spark-cluster-host:15002") \
    .appName("ConnectCapstone") \
    .getOrCreate()

# Start the long-running write
ranked_df.write.mode("overwrite").partitionBy("city").parquet("/output/rankings")

# If the client disconnects here, the write continues on the cluster
# Reconnect later and inspect the output
spark_reconnect = SparkSession.builder \
    .remote("sc://spark-cluster-host:15002") \
    .getOrCreate()

result = spark_reconnect.read.parquet("/output/rankings")
result.count()  # Confirm the write completed
```

**Important limitations to be aware of:**
- `SparkContext` is not available via Spark Connect — RDD API cannot be used.
- `sc.setLogLevel()` is not available; log level must be set server-side.
- Active streaming queries started from the client will terminate if the client process exits (unlike batch jobs).

**Expected outcome:**
- The batch Parquet write job completes on the cluster even if the client disconnects.
- The analyst reconnects with a new `SparkSession.builder.remote(...)` and reads the output successfully.
- This demonstrates the resilience benefit of Spark Connect's client-server separation.

**Exam topic reinforced:** Spark Connect — architecture, client-server separation, `sc://` connection syntax, `SparkContext` unavailability.

</details>

---
## Capstone Quick-Reference Cheat Sheet

```python
# ===== SCHEMA / CREATE =====
schema = StructType([StructField("col", StringType(), True)])
df = spark.createDataFrame(data, schema=schema)

# ===== INSPECT =====
df.printSchema()                       # show column names + types
df.show(5, truncate=False)             # show first N rows
df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

# ===== TYPE CASTING =====
df.withColumn("ts", F.to_timestamp("ts_str", "yyyy-MM-dd HH:mm:ss"))
df.withColumn("amt", F.col("amt_str").cast(DoubleType()))

# ===== NULL HANDLING =====
df.fillna({"col": default_value})
df.dropna(subset=["col"])

# ===== FILTER / SORT =====
df.filter(F.col("status") == "completed")
df.orderBy(F.col("fare").desc())

# ===== AGGREGATIONS =====
df.groupBy("key").agg(F.count("id"), F.sum("val"), F.avg("val"))

# ===== WINDOW FUNCTIONS =====
w = Window.partitionBy("city").orderBy(F.col("revenue").desc())
df.withColumn("rank", F.dense_rank().over(w))

# ===== UDF =====
my_udf = udf(lambda x: "val" if x is not None else None, StringType())
df.withColumn("new_col", my_udf(F.col("old_col")))

# ===== TEMP VIEW + SQL =====
df.createOrReplaceTempView("my_view")
spark.sql("SELECT * FROM my_view WHERE ...").show()

# ===== BROADCAST JOIN =====
large_df.join(broadcast(small_df), on="key", how="left")

# ===== PARQUET WRITE / READ =====
df.write.mode("overwrite").partitionBy("city").parquet("/path")
spark.read.parquet("/path").filter(F.col("city") == "NYC")  # partition pruning

# ===== STREAMING =====
q = df.writeStream.format("console").outputMode("complete").start()
q.stop()
```